In [190]:
import warnings
warnings.filterwarnings('ignore')

In [191]:
import pandas as pd
import numpy as np
import pyreadr
import joblib

import matplotlib.pyplot as plt
import plotly.express as px

from tqdm import tqdm

In [192]:
dem_uncont = pd.read_csv('transformed/dem_uncontested_seats.csv')
rep_uncont = pd.read_csv('transformed/rep_uncontested_seats.csv')

In [193]:
seat_sims = pyreadr.read_r('model_output/tot_seats_sims.RDS')[None]
seat_sims = seat_sims.rename({None: 'seats'}, axis=1)
seat_sims['seats'] = seat_sims['seats'].map(lambda x: x + dem_uncont.shape[0])
seat_sims['winner'] = seat_sims['seats'].map(lambda x: 'Democrats' if x >= 218 else 'Republicans')
seat_sims

,seats,winner
0,221,Democrats
1,253,Democrats
2,214,Republicans
3,233,Democrats
4,235,Democrats
...,...,...
19995,202,Republicans
19996,271,Democrats
19997,210,Republicans
19998,239,Democrats


In [194]:
sim_counts = seat_sims.groupby(['seats']).count().reset_index().rename({'winner': 'count'}, axis=1)
n_sims = seat_sims.shape[0]
sim_counts['pct'] = sim_counts['count'] / n_sims * 100
sim_counts['winner'] = sim_counts['seats'].map(lambda x: 'Democrats' if x >= 218 else 'Republicans')
seat_sims = pd.merge(left=seat_sims.drop(['winner'], axis=1), right=sim_counts, on='seats', how='left')
def get_desc(winner, pct, seats):
    return f'{winner} wins {seats if seats >= 218 else (435-seats)} seats in {pct:.2f}% of simulations'
#seat_sims['desc'] = seat_sims[['winner', 'pct', 'seats']].apply(lambda x: get_desc(x['winner'], x['pct'], x['seats']), axis=1)
sim_counts.head()

,seats,count,pct,winner
0,106,1,0.005,Republicans
1,144,1,0.005,Republicans
2,145,1,0.005,Republicans
3,152,1,0.005,Republicans
4,153,2,0.010,Republicans


In [195]:
np.unique(seat_sims['seats']).shape[0]

198

In [196]:
sims_hist = px.histogram(seat_sims, x='seats', nbins=np.unique(seat_sims['seats']).shape[0]*2, color='winner', 
                         labels={'seats':'Seats won by Democrats', 'winner': 'Winner'})
sims_hist.update_traces(showlegend=False)
sims_hist.add_vline(x=217.5, line_width=1, line_color='black', annotation_text='218 seats required for majority', 
                    annotation_position='top right')
sims_hist

In [197]:
# posterior prediction
post = pyreadr.read_r('model_output/labeled_posterior.RDS')[None]
post_untransp = post.copy()

In [198]:
post = post.T
post.head()

,0,1,2,3,4,5,6,7,8,9,...,19990,19991,19992,19993,19994,19995,19996,19997,19998,19999
AK-AL,-4.366518,-3.014024,-7.286766,-4.437377,-1.648926,-9.667486,-0.619480,-5.840689,-3.591817,-5.256302,...,-4.148507,-4.187977,-4.431311,-4.303894,-4.103900,-8.324025,-3.099684,-7.445949,-0.758228,-0.063730
AL-01,-14.983825,-10.383133,-17.874043,-14.419000,-13.030541,-13.340249,-17.997992,-12.506898,-17.523268,-12.922510,...,-15.242530,-19.947488,-18.717910,-12.878616,-11.773886,-15.609992,-11.214129,-13.544550,-15.032367,-16.314995
AL-02,-3.981734,-1.115693,-2.661204,-4.352910,0.013550,-5.687197,-1.568295,-4.804558,-6.609274,1.797156,...,-6.124517,-5.004266,-4.732515,-4.764443,-6.334146,-9.010591,2.109004,-7.091400,-1.411106,-3.181535
AL-03,-20.071020,-19.390597,-23.038109,-19.783675,-18.814347,-22.035889,-21.996397,-20.690652,-19.215847,-16.457049,...,-23.439134,-24.674275,-26.575785,-19.412348,-18.468526,-27.086481,-15.600820,-21.561222,-19.473522,-22.813299
AL-04,-31.942829,-26.506966,-35.068209,-36.799296,-26.224160,-34.561584,-29.850446,-31.364317,-27.033765,-30.323896,...,-34.137443,-33.820660,-34.028266,-30.513188,-29.701919,-37.869550,-27.934016,-32.036018,-30.073992,-30.732950


In [199]:
post.shape

(423, 20000)

In [200]:
def get_tipping_point(sim):
    """
    :param sim: Series representing one posterior draw or "simulation"
    :type sim: pd.Series
    """
    seats_won_dem = np.sum(sim > 0)
    if seats_won_dem >= 218 - dem_uncont.shape[0]: # Democrats win House in this draw
        sim = sim.sort_values(ascending=True)
        won_seats = sim[sim > 0]
        seat_margin = seats_won_dem - (218 - dem_uncont.shape[0])
    else: # Republicans win House in this draw
        sim = sim.sort_values(ascending=False)
        won_seats = sim[sim < 0]
        seat_margin = (435 - seats_won_dem) - (218 - rep_uncont.shape[0])
    return won_seats.iloc[seat_margin - 1], won_seats.index[seat_margin - 1]

In [201]:
tipping_points = np.array([]) # tipping point for *each sim*

for i in tqdm(range(post.shape[1])):
    sim = post.iloc[:, i]
    _, tp_seat = get_tipping_point(sim)
    tipping_points = np.append(tipping_points, tp_seat)

tipping_points

100%|███████████████████████████████████████████████████████████████████████████| 20000/20000 [00:30<00:00, 645.59it/s]


array(['NY-17', 'NY-03', 'WI-01', ..., 'OH-07', 'VA-02', 'NC-09'],
      shape=(20000,), dtype='<U32')

In [202]:
data = pd.read_csv('../../model_output/house_predictions.csv')
data.head()

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",...,-6.393775,-18.764506,-1,-37.529012,45.244310,3.721868,9.895,1,37.901862,52.553418
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",...,-28.357453,-38.923019,0,-77.846039,35.983197,3.688013,0.010,2,28.780528,43.232176
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",...,-6.526586,-19.716169,1,-39.432338,47.339465,3.583225,23.005,3,40.284383,54.352156
3,3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",...,-39.125354,-45.710827,-1,-91.421653,29.520570,3.602348,0.000,4,22.438662,36.625293
4,4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",...,-59.417450,-47.132870,-1,-94.265741,19.766532,3.625053,0.000,5,12.632904,26.915074


In [203]:
data['tipping_point_prob'] = data['cd'].map(lambda x: np.mean(tipping_points == x) * 100)
data.head()

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
0,0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0,"HILL, BILL","BEGICH, NICHOLAS III",...,-18.764506,-1,-37.529012,45.244310,3.721868,9.895,1,37.901862,52.553418,0.115
1,1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",...,-38.923019,0,-77.846039,35.983197,3.688013,0.010,2,28.780528,43.232176,0.000
2,2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2,"FIGURES, SHOMARI C.","MARQUES, RHETT",...,-19.716169,1,-39.432338,47.339465,3.583225,23.005,3,40.284383,54.352156,0.910
3,3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",...,-45.710827,-1,-91.421653,29.520570,3.602348,0.000,4,22.438662,36.625293,0.000
4,4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",...,-47.132870,-1,-94.265741,19.766532,3.625053,0.000,5,12.632904,26.915074,0.000


In [204]:
data.sort_values('tipping_point_prob', ascending=False).head(7)

,...1,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,...,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
233,233,NC-11,Jamie Ager,Jennifer Balkcom,False,False,NC,11,"AGER, JAMIE",no_match,...,50.000000,0,100.000000,50.896312,3.750283,59.550,234,43.520690,58.232055,3.155
197,197,MI-10,Christina Hines,Mike Bouchard,False,False,MI,10,"HINES, CHRISTINA","BOUCHARD, MICHAEL",...,-3.195838,0,-6.391676,50.436379,3.704983,54.880,198,43.021055,57.744981,3.135
37,37,CA-22,Randy Villegas,David Valadao,False,True,CA,22,"VILLEGAS, RANDY","VALADAO, DAVID",...,12.732090,-1,25.464181,50.391646,3.549305,54.735,38,43.394188,57.403636,2.735
191,191,MI-04,Sean McCann,Bill Huizenga,False,True,MI,4,"MCCANN, SEAN","HUIZENGA, WILLIAM P",...,7.674047,-1,15.348095,49.998911,3.533745,49.970,192,43.020911,56.969427,2.660
319,319,PA-07,Bob Brooks,Ryan Mackenzie,False,True,PA,7,"BROOKS, BOB","MACKENZIE, RYAN EDWARD",...,6.533790,-1,13.067581,50.251443,3.639258,52.810,320,42.999612,57.362769,2.645
278,278,NY-17,Cait Conley,Mike Lawler,False,True,NY,17,"CONLEY, CAIT","LAWLER, MICHAEL VINCENT",...,3.773526,-1,7.547052,50.629630,3.591514,57.080,279,43.596438,57.676862,2.645
187,187,ME-02,Matt Dunlap,Paul LePage,False,False,ME,2,"DUNLAP, MATT","LEPAGE, PAUL",...,-4.567471,0,-9.134942,49.471294,3.742267,44.070,188,42.040871,56.841742,2.500
